In [ ]:
from fastai.vision.all import *
from torch.nn import *


class Dat:
    a = torch.from_numpy(np.memmap(f"../tmp/news2016zhMin100.H", "h"))
    m = np.array([open("../tmp/news2016zhMin100.txt").read()]).view("U1")

    def __init__(s, w, b, n=1):
        s.a = s.a.as_strided([len(s.a) - w + 1, w], [1, 1])
        s.b = b
        s.n = n

    def __len__(s):
        return len(s.a) // (s.a.size(1) * s.b * s.n)

    def __iter__(s):
        for x in range(len(s)):
            x = s.a[torch.randint(len(s.a), [s.b])].to("cuda", torch.long)
            yield x, x


class Transpose(Module):
    @staticmethod
    def forward(x):
        return x.transpose(1, 2)


class ConvBn(Sequential):
    def __init__(_, i, w=3, s=1):
        super().__init__(Conv1d(i, i, w, s, w // 2, bias=0), BatchNorm1d(i, track_running_stats=0), Swish())


class ConvTransBn(Sequential):
    def __init__(_, i, w=3, s=1):
        super().__init__(ConvTranspose1d(i, i, w, s, w // 2, s // 2, bias=0), BatchNorm1d(i, track_running_stats=0), Swish())


class Res(Sequential):
    def forward(_, x):
        return super().forward(x) + x


class Cnn(Sequential):
    def __init__(s, *a, n=0):
        super().__init__(*a)
        s.n = n

    def forward(s, x):
        for _ in range(s.n or int(math.log2(x.size(2)))):
            x = super().forward(x)
        return x


class PyramidAutoEncoder(Sequential):
    def forward(s, x):
        y = [x]
        while x.size(2) > 1:
            x = s[0](x)
            y.append(x)
        y = list(s[1](torch.cat(y, 2)).split([y.size(2) for y in y], 2))
        x = y.pop()
        while y:
            x = torch.cat((y.pop(), s[2](x)))
        return x


class PyramidCrossEntropyLoss(CrossEntropyLoss):
    def forward(s, x, y):
        x = x.view(-1, len(y), *x.shape[1:])
        return super().forward(x.transpose(1, 2), y.broadcast_to(len(x), *y.shape))


def PyramidAccuracy(n):
    F = []
    for n in range(n):

        def f(x, y, n=n):
            with torch.no_grad():
                return accuracy(x.view(-1, len(y), *x.shape[1:])[n], y, 1)

        f.__name__ = str(n)
        F.append(f)
    return F

In [ ]:
m = Learner(
    DataLoaders(Dat(256, 32, 100), Dat(256, 32, 50000)),
    Sequential(
        Embedding(684, 256),
        Transpose(),
        PyramidAutoEncoder(
            Sequential(Res(ConvBn(256), ConvBn(256)), AvgPool1d(2)),
            Sequential(ConvBn(256, 1), Conv1d(256, 256, 1, bias=0)),
            Sequential(Res(ConvTransBn(256), ConvTransBn(256)), ConvTranspose1d(256, 256, 3, 2, 1, 1, bias=0)),
        ),
        Sequential(ConvBn(256, 1), Conv1d(256, 684, 1)),
    ).cuda(),
    PyramidCrossEntropyLoss(),
    cbs=[
        ShowGraphCallback,
        SaveModelCallback(fname="model5-autoencoder-Copy19b", with_opt=1, reset_on_fit=0),
    ],
    metrics=PyramidAccuracy(9),
    wd=0.01,
)
m.lr_find()
# m.load("model5-autoencoder-Copy19")
# m.validate()

In [ ]:
m.fit_one_cycle(100, .003, 9, 99, .01)

In [ ]:
m.fine_tune(300)